# 07 决策树 Decision Tree

决策树通过一连串“如果某个特征小于某个阈值，就往左，否则往右”的规则做预测。它可解释性强，也容易过拟合。


## 0. 学习目标和阅读地图

决策树是理解树模型和集成模型的基础。你需要掌握：

1. 节点纯度是什么意思。
2. 树为什么使用贪心切分。
3. 深度、叶子样本数如何控制复杂度。
4. 单棵树为什么容易过拟合。


## 1. 数学逻辑

分类树常用 Gini impurity 衡量一个节点有多混杂：

$$Gini = 1 - \sum_k p_k^2$$

如果一个节点里全是同一类，Gini 为 0。每次分裂时，决策树寻找能让子节点更纯的特征和阈值：

$$Gain = Gini(parent) - \frac{n_L}{n}Gini(left) - \frac{n_R}{n}Gini(right)$$


## 1.1 推导拆开看：一次切分为什么有效

一个节点越“纯”，说明里面的样本越接近同一类。Gini impurity 是：

$$Gini=1-\sum_k p_k^2$$

如果二分类节点里正负各一半，Gini 为：

$$1-(0.5^2+0.5^2)=0.5$$

如果全是同一类，Gini 为 0。

决策树每次尝试很多 `feature <= threshold` 的切分，选择让加权 Gini 降低最多的一刀。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

np.random.seed(42)
X, y = make_moons(n_samples=240, noise=0.25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)


## 1.2 决策树在二维数据上的形状

因为每次切分都是 `某个特征 <= 阈值`，二维里的决策边界通常是横平竖直的阶梯形。

树越深，阶梯越细，越能贴合训练数据；但太深时很容易把噪声也学进去。


In [ ]:
# 从零看一个节点如何选择最佳切分

def gini(labels):
    _, counts = np.unique(labels, return_counts=True)
    probs = counts / counts.sum()
    return 1 - np.sum(probs ** 2)

def best_stump_split(X, y):
    base = gini(y)
    best = {'gain': -1}
    for feature in range(X.shape[1]):
        for threshold in np.unique(X[:, feature]):
            left = X[:, feature] <= threshold
            if left.sum() == 0 or left.sum() == len(y):
                continue
            gain = base - left.mean() * gini(y[left]) - (~left).mean() * gini(y[~left])
            if gain > best['gain']:
                best = {'feature': feature, 'threshold': threshold, 'gain': gain}
    return best

print('根节点最佳一刀:', best_stump_split(X_train, y_train))


## 1.3 从零实现代码怎么读

`best_stump_split` 只找根节点的一刀，它已经包含了完整决策树的核心思想：

1. 枚举每个特征。
2. 枚举可能阈值。
3. 计算切分前后的 Gini 改善。
4. 选收益最大的切分。

完整决策树就是对左右子节点递归重复这个过程。


In [ ]:
model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('accuracy:', round(accuracy_score(y_test, pred), 3))

plt.figure(figsize=(10, 5))
plot_tree(model, feature_names=['x1', 'x2'], class_names=['0', '1'], filled=True, rounded=True)
plt.title('深度为 3 的决策树')
plt.show()

xx, yy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 180),
                     np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 180))
grid = np.c_[xx.ravel(), yy.ravel()]
zz = model.predict(grid).reshape(xx.shape)
plt.contourf(xx, yy, zz, alpha=0.25, cmap='coolwarm')
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, cmap='coolwarm', edgecolor='k', s=25)
plt.title('决策树的阶梯状决策边界')
plt.show()


In [ ]:
# 诊断：树深度如何影响训练/测试准确率
depths = range(1, 11)
train_acc, test_acc = [], []
for depth in depths:
    m_tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m_tree.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, m_tree.predict(X_train)))
    test_acc.append(accuracy_score(y_test, m_tree.predict(X_test)))

plt.plot(list(depths), train_acc, marker='o', label='train')
plt.plot(list(depths), test_acc, marker='o', label='test')
plt.title('树深度与过拟合')
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.legend()
plt.show()


## 2.1 如何诊断决策树

决策树最常见的问题是过拟合。典型信号是训练集准确率接近 1，但测试集明显更低。

常用约束：

- `max_depth`：限制最大深度。
- `min_samples_leaf`：叶子节点至少多少样本。
- `min_samples_split`：节点至少多少样本才允许继续切。


## 2. 常见误区

- 不限制深度的树很容易把训练集记住。
- 单棵树对数据扰动敏感，小变化可能导致结构变化很大。
- 特征重要性不一定等于因果重要性。

## 3. 小实验

- 改 `max_depth`，观察边界从简单到复杂。
- 改 `min_samples_leaf`，看过拟合是否缓解。
- 对比随机森林，观察稳定性提升。


## 5. 复习清单

- 决策树通过贪心切分降低节点混杂度。
- 单棵树可解释但不稳定。
- 深树容易过拟合，浅树容易欠拟合。
- 随机森林和 boosting 都建立在树模型基础上。
